# Superstore Sales EDA: 02. Feature Engineering & Customer RFM
---
**Objective:** Derive rich financial, temporal, operational, and customer behavioral metrics from the cleaned dataset.

### Engineered Dimensions:
1. **Financial:** Profit Margin %, Effective Unit Price, Effective Unit Cost, Is_Loss flag.
2. **Operational:** Shipping Lead Time in days (`Shipping_Days`).
3. **Temporal:** Year, Month, Month Name, Quarter, Day of Week, Year-Month period.
4. **Commercial:** Discount Buckets (`0% (None)`, `1-20% (Low)`, `21-50% (Medium)`, `50%+ (High)`).
5. **Customer Analytics (RFM):** Recency, Frequency, Monetary Value, and RFM Customer Segments.


In [1]:
import os
from pathlib import Path
import numpy as np
import pandas as pd

def get_path(rel_path):
    for candidate in [Path(rel_path), Path("..") / rel_path, Path("/home/adarsh/Documents/Superstore Sales EDA") / rel_path]:
        if candidate.exists():
            return candidate
    return Path(rel_path)

clean_path = get_path("data/cleaned/superstore_cleaned.csv")
df = pd.read_csv(clean_path)
df["Order Date"] = pd.to_datetime(df["Order Date"])
df["Ship Date"] = pd.to_datetime(df["Ship Date"])
print(f"Loaded cleaned dataset with {df.shape[0]:,} rows and {df.shape[1]} columns.")


Loaded cleaned dataset with 9,994 rows and 21 columns.


## 1. Financial & Operational Feature Engineering
Calculate Profit Margin (with zero division safeguard), Unit Financials, and Lead Times.


In [2]:
# Profit Margin %
df["Profit_Margin"] = np.where(df["Sales"] > 0, df["Profit"] / df["Sales"], 0.0)

# Unprofitable Order Flag
df["Is_Loss"] = df["Profit"] < 0

# Unit Economics
df["Effective_Unit_Price"] = df["Sales"] / df["Quantity"]
df["Effective_Unit_Cost"] = (df["Sales"] - df["Profit"]) / df["Quantity"]

# Shipping Lead Time (Calendar Days)
df["Shipping_Days"] = (df["Ship Date"] - df["Order Date"]).dt.days

print("Financial & Operational Feature Summary:")
df[["Profit_Margin", "Shipping_Days", "Effective_Unit_Price", "Effective_Unit_Cost"]].describe().T


Financial & Operational Feature Summary:


,count,mean,std,min,25%,50%,75%,max
Profit_Margin,9994.0,0.120314,0.466754,-2.7500,0.0750,0.2700,0.3625,0.5000
Shipping_Days,9994.0,3.958175,1.747567,0.0000,3.0000,4.0000,5.0000,7.0000
Effective_Unit_Price,9994.0,60.919569,142.927440,0.3360,5.4700,16.2700,63.9400,3773.0800
Effective_Unit_Cost,9994.0,53.120197,122.251398,0.5445,3.4736,12.9336,54.5220,4074.9264


## 2. Temporal Decomposition
Extract calendar dimensions to support monthly seasonality, quarterly performance, and day-of-week analysis.


In [3]:
df["Order_Year"] = df["Order Date"].dt.year
df["Order_Month"] = df["Order Date"].dt.month
df["Order_Month_Name"] = df["Order Date"].dt.strftime("%B")
df["Order_Quarter"] = "Q" + df["Order Date"].dt.quarter.astype(str)
df["Order_Day_Name"] = df["Order Date"].dt.day_name()
df["Year_Month"] = df["Order Date"].dt.to_period("M").astype(str)

print("Orders per Year:")
print(df["Order_Year"].value_counts().sort_index())


Orders per Year:
Order_Year
2014    1993
2015    2102
2016    2587
2017    3312
Name: count, dtype: int64


## 3. Discount Bucketing
Discounts in Superstore range from 0% to 80%. We categorize them into actionable business tiers:
- **0% (None):** Full retail price transactions
- **1-20% (Low):** Standard retail promotions
- **21-50% (Medium):** Aggressive promotional discounting
- **50%+ (High):** Heavy clearance discounting


In [4]:
def categorize_discount(d: float) -> str:
    if d == 0.0:
        return "0% (None)"
    elif d <= 0.20:
        return "1-20% (Low)"
    elif d <= 0.50:
        return "21-50% (Medium)"
    else:
        return "50%+ (High)"

df["Discount_Bucket"] = df["Discount"].apply(categorize_discount)
bucket_order = ["0% (None)", "1-20% (Low)", "21-50% (Medium)", "50%+ (High)"]
df["Discount_Bucket"] = pd.Categorical(df["Discount_Bucket"], categories=bucket_order, ordered=True)

df["Discount_Bucket"].value_counts().sort_index()


Discount_Bucket
0% (None)          4798
1-20% (Low)        3803
21-50% (Medium)     537
50%+ (High)         856
Name: count, dtype: int64

## 4. Customer RFM Segmentation
We synthesize Customer Recency (days since last purchase), Frequency (order count), and Monetary Value (lifetime revenue and profit).


In [5]:
snapshot_date = df["Order Date"].max() + pd.Timedelta(days=1)

rfm = df.groupby(["Customer ID", "Customer Name", "Segment"]).agg(
    Recency_Days=("Order Date", lambda x: (snapshot_date - x.max()).days),
    Frequency=("Order ID", "nunique"),
    Total_Items=("Row ID", "count"),
    Monetary_Sales=("Sales", "sum"),
    Monetary_Profit=("Profit", "sum"),
    Avg_Discount=("Discount", "mean"),
    Loss_Orders_Count=("Is_Loss", lambda x: int(x.sum())),
).reset_index()

rfm["Profit_Margin"] = np.where(
    rfm["Monetary_Sales"] > 0,
    rfm["Monetary_Profit"] / rfm["Monetary_Sales"],
    0.0
)

# Quintile Scoring (1 to 5)
rfm["R_Score"] = pd.qcut(rfm["Recency_Days"], q=5, labels=[5, 4, 3, 2, 1], duplicates="drop").astype(int)
rfm["F_Score"] = pd.qcut(rfm["Frequency"].rank(method="first"), q=5, labels=[1, 2, 3, 4, 5]).astype(int)
rfm["M_Score"] = pd.qcut(rfm["Monetary_Sales"].rank(method="first"), q=5, labels=[1, 2, 3, 4, 5]).astype(int)

def segment_customer(row) -> str:
    r, f, m = row["R_Score"], row["F_Score"], row["M_Score"]
    if r >= 4 and f >= 4 and m >= 4:
        return "Champions"
    elif f >= 3 and m >= 3:
        return "Loyal Customers"
    elif r >= 3 and f >= 2:
        return "Potential Loyalists"
    elif r >= 4 and f <= 2:
        return "Recent / New Customers"
    elif r <= 2 and (f >= 3 or m >= 3):
        return "At Risk / Needs Attention"
    elif r <= 2 and f <= 2 and m <= 2:
        return "Hibernating / Lost"
    else:
        return "Promising Customers"

rfm["RFM_Segment"] = rfm.apply(segment_customer, axis=1)

print("Customer RFM Segment Distribution:")
rfm["RFM_Segment"].value_counts()


Customer RFM Segment Distribution:


RFM_Segment
Loyal Customers              260
Potential Loyalists          150
Hibernating / Lost           117
Champions                    106
At Risk / Needs Attention     98
Recent / New Customers        39
Promising Customers           23
Name: count, dtype: int64

## 5. Enrich Transaction Layer & Persist Deliverables
Merge `RFM_Segment` back into order items and write both datasets to `data/cleaned/`.


In [6]:
df = df.merge(
    rfm[["Customer ID", "RFM_Segment"]],
    on="Customer ID",
    how="left"
)

out_dir = Path("data/cleaned") if Path("data").exists() else Path("../data/cleaned")
out_featured = out_dir / "superstore_featured.csv"
out_rfm = out_dir / "customer_rfm.csv"

df.to_csv(out_featured, index=False)
rfm.to_csv(out_rfm, index=False)

print(f"Exported feature-engineered dataset: {out_featured} ({df.shape})")
print(f"Exported customer RFM dataset:        {out_rfm} ({rfm.shape})")


Exported feature-engineered dataset: data/cleaned/superstore_featured.csv ((9994, 34))
Exported customer RFM dataset:        data/cleaned/customer_rfm.csv ((793, 15))
